# 实验：高斯消元法（列主元）

---

## 实验概述

本实验编译并运行用 C++ 实现的**高斯消元算法**（`gaussian_elimination.cpp` 与 `gaussian_elimination_simd.cpp`），通过观察程序输出分析算法的正确性和性能。

## 学习目标

完成本实验后，你将能够：

1. 说明高斯消元法的三个阶段：**列主元选取**、**前向消去**、**回代求解**
2. 解释**列主元策略**如何提高数值稳定性
3. 使用**无穷范数残差** $\|Ax - b\|_\infty$ 验证求解精度
4. 理解 **AVX2 SIMD** 如何加速消元内层循环
5. 对比标量版与 SIMD 版的性能差异

## 0. 检查 CPU 支持的 SIMD 指令集

在选择 SIMD 优化方案之前，先确认服务器 CPU 的指令集支持情况。

In [ ]:
import subprocess

# 查看 CPU 架构
arch = subprocess.run(["uname", "-m"], capture_output=True, text=True).stdout.strip()
print(f"CPU 架构: {arch}\n")

# 查看支持的 SIMD 指令集
result = subprocess.run(
    "grep -m1 'flags\\|Features' /proc/cpuinfo | tr ' ' '\\n' | grep -E 'neon|asimd|sse|avx|fma|mmx' | sort -u",
    shell=True, capture_output=True, text=True
)
print("支持的 SIMD 扩展指令集:")
print(result.stdout if result.stdout else "  （未检测到，或当前平台不支持 /proc/cpuinfo）")

## 1. 查看源代码

### 1.1 标量版（`gaussian_elimination.cpp`）

In [ ]:
with open("gaussian_elimination.cpp", "r", encoding="utf-8") as f:
    print(f.read())

### 1.2 AVX2 SIMD 版（`gaussian_elimination_simd.cpp`）

In [ ]:
with open("gaussian_elimination_simd.cpp", "r", encoding="utf-8") as f:
    print(f.read())

## 2. 算法原理

高斯消元法分三个阶段求解 $Ax = b$：

### 第一阶段 — 列主元选取（每列 $k$）

1. 找 $p = \arg\max_{i \geq k} |a_{ik}|$，即当前列绝对值最大的行
2. 若 $|a_{pk}| < \varepsilon$（$\varepsilon = 10^{-9}$）→ 矩阵奇异，终止
3. 将第 $k$ 行与第 $p$ 行交换

### 第二阶段 — 前向消去（每列 $k$，消去第 $i > k$ 行）

$$
m_{ik} = \frac{a_{ik}}{a_{kk}}, \qquad
a_{ij} \leftarrow a_{ij} - m_{ik} \cdot a_{kj}, \quad j = k, \ldots, n
$$

经过所有列后，$[A \mid b]$ 化为**上三角矩阵**。

### 第三阶段 — 回代求解

$$
x_i = \frac{b_i - \displaystyle\sum_{j=i+1}^{n-1} a_{ij}\, x_j}{a_{ii}}, \quad i = n-1, \ldots, 0
$$



## 3. 编译标量版程序

使用 `g++` 编译 `gaussian_elimination.cpp`，开启 `-O2` 优化与 C++17 标准。

In [ ]:
import subprocess, os, sys

SRC    = "gaussian_elimination.cpp"
EXE    = "gaussian_elimination" + (".exe" if sys.platform == "win32" else "")

result = subprocess.run(
    ["g++", "-O2", "-std=c++17", "-o", EXE, SRC],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"[OK] 编译成功: {SRC} -> {EXE}")
else:
    print("[失败] 编译错误:")
    print(result.stderr)

## 4. 运行标量版 — 示例一：4×4 方程组

内置示例一使用如下增广矩阵（精确解为 $x = [1,2,3,4]^T$）：

$$
[A \mid b] =
\begin{bmatrix}
4 & 1 & 2 & 3 & \mid & 24 \\
3 & 4 & 1 & 2 & \mid & 22 \\
2 & 3 & 4 & 1 & \mid & 24 \\
1 & 2 & 3 & 4 & \mid & 30
\end{bmatrix}
$$

向 stdin 传入 `0` 以跳过交互式自定义输入。

In [ ]:
result = subprocess.run(
    [os.path.join(".", EXE)],
    input="0\n",
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## 5. 运行标量版 — 示例二：Hilbert 病态矩阵

**Hilbert 矩阵** $H_{ij} = \dfrac{1}{i+j+1}$ 是经典的病态矩阵，条件数随 $n$ 急剧增大。  
程序构造 $b = H \cdot \mathbf{1}$，因此精确解为 $x = [1,1,1]^T$。

| $n$ | 条件数 $\kappa(H_n)$（近似） |
|:----|----------------------------:|
| 3   | $5.2 \times 10^{2}$         |
| 5   | $4.8 \times 10^{5}$         |
| 8   | $1.5 \times 10^{10}$        |


In [ ]:
# 示例二已内置于程序中，同样传入 0 跳过自定义输入
result = subprocess.run(
    [os.path.join(".", EXE)],
    input="0\n",
    capture_output=True, text=True, encoding="utf-8"
)
# 只打印示例二相关输出
lines = result.stdout.splitlines()
in_ex2 = False
for line in lines:
    if "Example 2" in line or "Hilbert" in line:
        in_ex2 = True
    if in_ex2:
        print(line)
    if in_ex2 and line.startswith("---") and "Example 3" in line:
        break

## 6. 运行标量版 — 自定义输入

自定义输入格式：
```
n
a[0][0] a[0][1] ... a[0][n-1] b[0]
...
a[n-1][0] ...           b[n-1]
```

以下示例求解 3×3 方程组（精确解 $x = [2, 3, -1]^T$）：

$$
\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \\ x_2 \end{bmatrix}
=
\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}
$$

In [ ]:
custom_input = """\
3
2 1 -1 8
-3 -1 2 -11
-2 1 2 -3
"""
result = subprocess.run(
    [os.path.join(".", EXE)],
    input=custom_input,
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## 7. 奇异矩阵检测

当矩阵奇异（某列主元绝对值 $< \varepsilon = 10^{-9}$）时，程序应报错终止。  
下面以行 2 = 行 0 + 行 1 的秩亏矩阵为例进行验证。

In [ ]:
# 奇异矩阵：行2 = 行0 + 行1（秩亏）
singular_input = """\
3
1 2 3 6
4 5 6 15
5 7 9 21
"""
result = subprocess.run(
    [os.path.join(".", EXE)],
    input=singular_input,
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

---

## 8. SIMD 原理介绍

### 8.1 什么是 SIMD？

**SIMD（Single Instruction, Multiple Data，单指令多数据流）** 是一种数据级并行技术：  
一条指令同时对多个数据元素执行相同的操作，从而在不增加时钟频率的前提下提高吞吐量。

```
标量（Scalar）操作：
  a0 + b0 = c0          ← 一条指令，处理 1 个 double

SIMD（AVX2）操作：
  [a0, a1, a2, a3]
+
  [b0, b1, b2, b3]
=
  [c0, c1, c2, c3]      ← 一条指令，同时处理 4 个 double
```

### 8.2 x86 SIMD 指令集演进

| 指令集 | 寄存器宽度 | 单精度 (float) 路数 | 双精度 (double) 路数 | 引入年份 |
|:-------|:----------:|:-------------------:|:--------------------:|:--------:|
| SSE2   | 128 bit    | 4                   | 2                    | 2001     |
| AVX    | 256 bit    | 8                   | 4                    | 2011     |
| **AVX2**  | **256 bit** | **8**             | **4**                | **2013** |
| AVX-512 | 512 bit   | 16                  | 8                    | 2016     |

本实验服务器支持 **AVX2 + AVX-512**，我们使用 AVX2（`__m256d`）实现，兼容性更广泛。

### 8.3 AVX2 关键数据类型与 intrinsic 函数

| 类型 / 函数 | 说明 |
|:------------|:-----|
| `__m256d` | 256 位寄存器，存放 **4 个 `double`** |
| `_mm256_set1_pd(x)` | 将标量 `x` 广播到 4 个通道：`[x, x, x, x]` |
| `_mm256_loadu_pd(ptr)` | 从内存非对齐加载 4 个 `double` |
| `_mm256_storeu_pd(ptr, v)` | 将 4 个 `double` 非对齐写回内存 |
| `_mm256_fmadd_pd(a, b, c)` | **融合乘加（FMA）**：`a*b + c`，单指令，精度更高 |

### 8.4 融合乘加（FMA）

普通方式需要两条指令，且中间结果有舍入误差：
$$
c \leftarrow c + a \times b \quad \Rightarrow \quad \text{mul} + \text{add（两次舍入）}
$$

FMA 在硬件中一次完成，**只舍入一次**：
$$
\texttt{fmadd}(a, b, c) = a \times b + c \quad \text{（单次舍入，精度更高）}
$$

在消元循环中，我们用 $a = {-\text{factor}}$，$b = \text{pivot\_row}[k]$，$c = \text{row}[k]$，即：
$$
\text{row}[k] \leftarrow \text{row}[k] + (-\text{factor}) \times \text{pivot\_row}[k]
$$

### 8.5 内存对齐

AVX2 的对齐加载指令（`_mm256_load_pd`）要求地址为 **32 字节对齐**。  
本实验的 SIMD 版本使用 `posix_memalign` 分配 32 字节对齐的矩阵，并将行步长（leading dimension）取为 **4 的倍数**，保证每行起始地址均满足对齐要求。

```
行 0: [a00, a01, a02, a03, a04, ..., b0, pad, pad, pad]  ← 32字节对齐
行 1: [a10, a11, ...]                                     ← 32字节对齐
...
```

### 8.6 标量 vs SIMD 消元循环对比

```cpp
// ── 标量版 ─────────────────────────────────────────────
for (int k = col+1; k <= n; ++k)
    row[k] -= factor * pivot_row[k];   // 每次处理 1 个 double

// ── AVX2 SIMD 版 ────────────────────────────────────────
__m256d vf = _mm256_set1_pd(-factor);  // 广播 -factor 到 4 通道
int k = col + 1;
for (; k + 4 <= n + 1; k += 4) {
    __m256d vr = _mm256_loadu_pd(row + k);
    __m256d vp = _mm256_loadu_pd(pivot_row + k);
    vr = _mm256_fmadd_pd(vf, vp, vr);  // 4 路 FMA，一次处理 4 个 double
    _mm256_storeu_pd(row + k, vr);
}
for (; k <= n; ++k)                    // 标量尾处理
    row[k] -= factor * pivot_row[k];
```

---

## 9. SIMD 优化：AVX2 向量化高斯消元

### 9.1 优化动机

消元内层循环是整个算法的性能瓶颈（$O(n^3)$ 次浮点运算集中于此）：

```cpp
for (int k = col+1; k <= n; ++k)
    row[k] -= factor * pivot_row[k];   // 每行 n 次 FMA，共 O(n²) 行
```

使用 **AVX2**，`__m256d` 寄存器一次携带 **4 个 `double`**，通过 `_mm256_fmadd_pd` 单指令完成 4 路融合乘加，理论上可将该循环吞吐量提升至 **4 倍**。

### 9.2 关键设计

| 设计决策 | 说明 |
|:---------|:-----|
| **对齐平铺存储** | 行主序一维数组，32 字节对齐，支持 `_mm256_load_pd` |
| **行步长为 4 的倍数** | 保证每行起始地址均在 32 字节边界 |
| **FMA 指令** | `vfnmadd231pd`：`dst = dst + (-factor) × src`，一条指令完成 |
| **行交换用 AVX2** | 256 位加载/存储整行，减少循环次数 |
| **标量尾处理** | 当 `(n+1)` 不是 4 的倍数时，用标量处理剩余元素 |

### 9.3 SIMD 消元核心代码

```cpp
__m256d vfactor = _mm256_set1_pd(-factor);   // 将 -factor 广播到 4 个通道
int k = col + 1;

// AVX2 主循环：每次处理 4 个 double
for (; k + 4 <= n + 1; k += 4) {
    __m256d vr = _mm256_loadu_pd(rr + k);        // 从目标行加载 4 个 double
    __m256d vp = _mm256_loadu_pd(pivot_rp + k);  // 从主元行加载 4 个 double
    vr = _mm256_fmadd_pd(vfactor, vp, vr);        // vr = vr + (-factor)*vp
    _mm256_storeu_pd(rr + k, vr);                 // 写回
}
// 标量尾处理
for (; k <= n; ++k)
    rr[k] -= factor * pivot_rp[k];
```

### 9.4 编译 SIMD 版（需加 `-mavx2 -mfma`）

In [ ]:
SRC_SIMD = "gaussian_elimination_simd.cpp"
EXE_SIMD = "gaussian_elimination_simd" + (".exe" if sys.platform == "win32" else "")

result = subprocess.run(
    ["g++", "-O2", "-std=c++17", "-mavx2", "-mfma", "-o", EXE_SIMD, SRC_SIMD],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"[OK] 编译成功: {SRC_SIMD} -> {EXE_SIMD}")
    print("编译选项: -O2 -std=c++17 -mavx2 -mfma")
else:
    print("[失败] 编译错误:")
    print(result.stderr)

### 9.5 运行 SIMD 版 — 验证正确性

In [ ]:
result = subprocess.run(
    [os.path.join(".", EXE_SIMD)],
    input="0\n",
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

### 9.6 性能对比：标量版 vs AVX2 版

用 Python 生成一个大规模随机线性系统（$n = 500$），分别传给两个可执行文件计时，对比墙上时间。

In [ ]:
import numpy as np
import time

def generate_input(n, seed=42):
    """生成对角占优的 n×n 随机线性系统，确保有唯一解。"""
    rng = np.random.default_rng(seed)
    A = rng.random((n, n))
    np.fill_diagonal(A, A.diagonal() + n)   # 对角占优 → 非奇异
    x_exact = rng.random(n)
    b = A @ x_exact
    lines = [str(n)]
    for i in range(n):
        row = " ".join(f"{A[i,j]:.10f}" for j in range(n)) + f" {b[i]:.10f}"
        lines.append(row)
    return "\n".join(lines) + "\n"

N = 500
stdin_data = generate_input(N)
print(f"已生成 {N}×{N} 线性系统（输入大小约 {len(stdin_data)//1024} KB）\n")

def run_and_time(exe, stdin_data, label):
    t0 = time.perf_counter()
    r = subprocess.run(
        [os.path.join(".", exe)],
        input=stdin_data,
        capture_output=True, text=True, encoding="utf-8"
    )
    elapsed = (time.perf_counter() - t0) * 1000
    if r.returncode != 0:
        print(f"[{label}] 错误: {r.stderr[:200]}")
        return
    for line in r.stdout.splitlines():
        if "Residual" in line:
            print(f"[{label}]  {line.strip()}   |  耗时: {elapsed:.1f} ms")
            break

run_and_time(EXE,      stdin_data, "标量版 ")
run_and_time(EXE_SIMD, stdin_data, "AVX2版 ")

### 9.7 实验记录

填写运行结果：

| 版本 | 编译选项 | 耗时 (ms) | 残差 $\|Ax-b\|_\infty$ |
|:-----|:--------|:---------:|:----------------------:|
| 标量版 | `-O2` | ______ | ______ |
| AVX2 版 | `-O2 -mavx2 -mfma` | ______ | ______ |
| 加速比 | — | ______ × | — |

